# NLLB corpus-quality experiments — four stages

**Default: Stage 1, untouched-model lowest baseline.** Generate translations for the entire development (eval) and test sets; save all predictions, full BLEU/chrF++ scores, plots and an independent ZIP. No finetuning is performed. Development/test comparisons are **descriptive**, never used for search. For Stage 2, switch just `stage` and `allow_training` in the settings cell. See `docs/four-stage-workflow.md`.


## 1. Editable experiment settings

This is the **only** cell you edit. Reimport the notebook and restart the runtime for a new experiment.


In [ ]:
# ONE EDITABLE CELL. Fresh kernel/reimport for each experiment.
EXPERIMENT = {
    # STAGES: lowest_baseline | low_baseline | decoding_search | language_extension
    "stage": "lowest_baseline",      # DEFAULT: untouched model, FULL dev+test inference.
    "run_label": "corpus-study-1",   # Same label across stages; new label for new runs.

    # DATA (edit file paths): two aligned .txt files, or one .tsv/.jsonl.
    "data_mode": "aligned_text",     # aligned_text | tsv | jsonl | pre_split | example
    "source_lang": "bho_Deva",      # Use a native NLLB tag in Stages 1–3.
    "target_lang": "hin_Deva",
    "source_text": "/kaggle/input/YOUR_DATASET/bho.txt",
    "target_text": "/kaggle/input/YOUR_DATASET/hin.txt",
    "tsv_file": "/kaggle/input/YOUR_DATASET/parallel.tsv",
    "tsv_has_header": False,
    "parallel_jsonl": "data/local/bho-hin/parallel.jsonl",
    "split_directory": "data/local/bho-hin/splits",
    "dataset_id": "corpus-v1",     # Change if you replace/change the corpus.
    "split_seed": 42,
    "split_ratios": (0.80, 0.10, 0.10),
    "group_by_document": True,
    "validate_data": False,
    "reject_within_split_duplicates": False,  # Allow repeated corpus pairs; ALWAYS block cross-split leakage.       # False = quick start; true checks data leakage fully.
    "data_report": False,         # Extra corpus diagnostics; independent of metrics.

    # MODEL AND GENERATION. Stage 1/2 use the SAME fixed decoding.
    "model_id": "facebook/nllb-200-distilled-600M",
    "model_revision": None,       # Optional immutable HF commit hash.
    "conditioning": "native",     # No new language tags in Stages 1–3.
    "beams": 4,
    "length_penalty": 1.0,
    "max_new_tokens": 128,
    "max_length": 128,            # Long sentences can be truncated (inspect later).
    "batch_size": 1,

    # STAGE 2 / 4 ONLY: true training, with development loss every epoch.
    "allow_training": False,      # Set True for low_baseline / language_extension.
    "finetune_mode": "full",       # Full model; choose "lora" if GPU memory is limited.
    "train_seed": 42,
    "epochs": 3, "max_steps": -1, # -1 = all epochs; 1 = software smoke test.
    "gradient_accumulation_steps": 16,
    "learning_rate": 2e-5,        # Typical cautious full-model setting.
    "precision": "fp16",         # T4: fp16; bf16 only on a supported GPU.
    "gradient_checkpointing": True,
    "lora_rank": 8,               # Used only when finetune_mode = "lora".

    # STAGE 3 ONLY: select beam/length penalty on DEV, then test once.
    "search_source_stage": "low_baseline",
    "search_metric": "chrfpp",    # chrfpp | bleu
    "search_grid": [
        {"beams": 1, "length_penalty": 1.0},
        {"beams": 4, "length_penalty": 1.0},
        {"beams": 6, "length_penalty": 1.0},
    ],

    # STAGE 4 ONLY: extension requires a new language tag + full training.
    "extend_language": None,     # e.g. "bra_Deva"; set conditioning="dedicated_token".
    "language_donor": "hin_Deva",
    "proxy_source_lang": None,    # Only for separate text-prefix experiments.
    "proxy_target_lang": None,

    # REPORTS + SAVING: work in ALL four stages, even with no finetuning.
    "result_plots": True,        # SVG charts, metrics, and full output review CSV.
    "save_archive": True,        # Auto-ZIP current stage on success or failure.
    "include_checkpoints": False,# True = also keep large optimizer state.
    "setup_mode": "train",       # Runtime with GPU and translation dependencies.
    "refresh_repo": False,      # True only when pulling an existing checkout.

    # OPTIONAL CLEANUP. Leave "off" normally. Run ONLY settings+cleanup cells to use.
    "cleanup_action": "off",        # off | preview | delete
    "cleanup_target_stage": "low_baseline", # Stage to clean, even if currently in another stage.
    "cleanup_confirmation": "",    # Exact phrase printed by preview; required for delete.
}


## 2. Environment

Select GPU and Internet before running. Automatically clones the repo when absent.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json

# Install/clone AFTER the settings cell. No separate terminal is needed.
ROOT = (Path("/kaggle/working/BhaashikMTFinetuning")
        if Path("/kaggle/working").is_dir()
        else Path("/content/BhaashikMTFinetuning")
        if Path("/content").is_dir() else Path.cwd().resolve())
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
if not (ROOT / "pyproject.toml").is_file():
    if ROOT.exists():
        raise ValueError(f"Checkout exists but is incomplete: {ROOT}")
    subprocess.run(["git", "clone",
                    "https://github.com/bhaashik/BhaashikMTFinetuning.git",
                    str(ROOT)], check=True)
elif EXPERIMENT["refresh_repo"]:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
loaded = [x for x in ("torch", "transformers", "sentencepiece", "datasets", "peft")
          if x in sys.modules]
if loaded:
    raise RuntimeError("Restart session before dependency installation: "+", ".join(loaded))
setup_mode = EXPERIMENT["setup_mode"]
command = [sys.executable, str(ROOT / "scripts" / "setup_notebook.py"),
           "--extras", setup_mode]
if setup_mode.startswith("train"):
    command += ["--require-cuda"]
subprocess.run(command, check=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from bhaashik.experiment_workflow import check_settings
check_settings(EXPERIMENT, model_family="nllb")
print("Checkout:", ROOT, "| mode:", setup_mode)


## 3. Prepare real or example data

Convert aligned text/TSV, perform seeded train/dev/test splitting, record hashes. No model inference.


In [ ]:
from bhaashik.experiment_workflow import (
    prepare_parallel_data, experiment_paths, training_parameters,
    write_experiment_config, report_data_if_enabled,
)
PREPARED = prepare_parallel_data(ROOT, EXPERIMENT, model_family="nllb")
PATHS = experiment_paths(ROOT, EXPERIMENT, family="nllb", prepared=PREPARED)
DATA = PREPARED["data_dir"]
STAGE_DIR = PATHS["stage"]       # Stage-specific results, not necessarily a trained model.
report_data_if_enabled(EXPERIMENT, PREPARED, PATHS["report"])
TRAIN_CONFIG = training_parameters(EXPERIMENT, PREPARED, PATHS,
                                   model_family="nllb")
TRAIN_CONFIG_PATH = write_experiment_config(
    EXPERIMENT, PREPARED, PATHS, TRAIN_CONFIG)
print("Stage:", EXPERIMENT["stage"], "Results:", STAGE_DIR)


## 4. Stage overview

Shows whether this stage trains, or only evaluates a model.


In [ ]:
# This cell does not download a model or run inference.
stage = EXPERIMENT["stage"]
print("Experiment:", stage)
print("Source:", EXPERIMENT["source_lang"], "->", EXPERIMENT["target_lang"])
if stage == "lowest_baseline":
    print("Untouched NLLB: full DEV and TEST predictions, scores, plots, ZIP.")
elif stage == "low_baseline":
    print("Full training, per-epoch DEV LOSS, then full DEV and TEST reports.")
elif stage == "decoding_search":
    print("Evaluate decoding grid on DEV, select best, then evaluate TEST once.")
else:
    print("Extend language token, train embeddings, evaluate DEV and TEST.")
print("No training in this stage:", stage in ("lowest_baseline", "decoding_search"))


## 5. Quick safety checks

Full corpus diagnostics remain optional. Stage 2 still audits train/dev data.


In [ ]:
# This short precheck is separate from optional full corpus diagnostics.
from bhaashik.experiment_workflow import check_settings
check_settings(EXPERIMENT, model_family="nllb")
if EXPERIMENT["stage"] == "language_extension":
    assert EXPERIMENT["extend_language"], "Select new language tag in first cell."
    assert EXPERIMENT["finetune_mode"] == "full"
print("Configuration valid. Split counts:", PREPARED["counts"])


## 6. Execute this stage

Runs generation, full metrics, plots and auto-archives. Stage 2/4 also records validation loss.


In [ ]:
from bhaashik.staged_experiments import run_stage
# Runs one stage only. Stages 1/2/4 produce FULL dev+test predictions/metrics.
# Stage 3 searches DEV first and evaluates TEST only with its chosen decoding.
# A ZIP is saved automatically when save_archive=True, including partial results
# if a model operation fails. No finetuned checkpoint is needed for Stage 1.
RESULT = run_stage(ROOT, EXPERIMENT, PREPARED, PATHS, TRAIN_CONFIG_PATH,
                   family="nllb")
print("Completed:", RESULT["stage"])


## 7. Results summary

Scores and output paths for the stage that ran.


In [ ]:
import json
from bhaashik.data import file_hash
manifest = STAGE_DIR / "stage_manifest.json"
print("Stage status:", json.loads(manifest.read_text())["status"])
if (STAGE_DIR / "stage_results.json").is_file():
    saved = json.loads((STAGE_DIR / "stage_results.json").read_text())
    for split, key in (("DEV", "dev_scores"), ("TEST", "test_scores")):
        print(split, saved[key]["bleu"], "BLEU;",
              saved[key]["chrfpp"], "chrF++")
    print("Model:", saved["model"])
    print("Output files:", sorted(p.name for p in STAGE_DIR.iterdir()))
else:
    print("Stage incomplete; examine stage_manifest.json and preserved outputs.")


## 8. Saved archive

Independent of finetuning. Store the ZIP somewhere persistent before ending the session.


In [ ]:
# This check is independent of training completion.
# run_stage already auto-saved the ZIP on completion/exception when enabled.
print("Stage results:", STAGE_DIR)
archive = PATHS["archive"]
if archive.exists():
    print("Download/save:", archive, "size MiB:", round(archive.stat().st_size/2**20, 1))
else:
    print("No ZIP found. Check save_archive, the stage manifest and runtime disk.")
# Kaggle: Save Version / Output files; Colab: copy ZIP to mounted Google Drive.


## Optional cleanup of a mistaken experiment

**Run this cell separately, not as part of the normal experiment.** In the
first settings cell set `cleanup_action="preview"` and
`cleanup_target_stage` to the mistaken stage. Run only this cleanup
cell to see its files. For deletion, change `cleanup_action="delete"`
and paste the exact printed phrase into `cleanup_confirmation`, then
run this cell alone. It cannot delete source data, split manifests,
other stages, or shared report folders. Default is off.


In [ ]:
# OPTIONAL: clean an incorrect generated experiment, without touching source data.
# To clean: set cleanup_action="preview" in the FIRST settings cell and run
# this cell ONLY (after the settings cell); read the files it lists.
# To delete: set cleanup_action="delete", copy its exact confirmation phrase
# into cleanup_confirmation, and run THIS CELL ONLY. Do not click Run All.
# The selected stage, its ZIP and stage-specific configs are the only targets.
from pathlib import Path
from bhaashik.experiment_cleanup import cleanup_stage

if EXPERIMENT["cleanup_action"] == "off":
    print("Cleanup disabled. To use it, edit cleanup_* in the first settings cell.")
else:
    root_for_cleanup = (
        Path("/kaggle/working/BhaashikMTFinetuning")
        if Path("/kaggle/working").is_dir()
        else Path("/content/BhaashikMTFinetuning")
        if Path("/content").is_dir()
        else Path.cwd()
    )
    cleanup_plan = cleanup_stage(
        root_for_cleanup, EXPERIMENT,
        family="nllb",
        stage=EXPERIMENT["cleanup_target_stage"],
        action=EXPERIMENT["cleanup_action"],
        confirmation=EXPERIMENT["cleanup_confirmation"],
    )
    print("Deletion completed:", cleanup_plan["deleted"])
